# OmniVoice — Yoruba tone probe (zero-shot, NO training)

**Goal.** Decide go / no-go on pivoting the TTS base from **Qwen3-TTS** to
**k2-fsa/OmniVoice** for Yoruba — *cheaply*, before any finetune.

We zero-shot synthesize held-out Yoruba (tone **minimal pairs** + a few longer held-out lines)
with OmniVoice, voice-cloned from **one clean BibleTTS reference**, then score each wav with the
project's existing tone gate:

| metric | what | module |
|---|---|---|
| **I2 `tone_i2`** | real tone (F0-absolute H/M/L) — *the decisive signal* | `tone_metric/tone_f0_abs.py` |
| **I1 `i1_acc`** | quality / anti-collapse (AfriHuBERT probe) | `tone_metric/tone_probe.py` |
| **CER** | intelligibility (MMS-1b-all `yor`) — **tone-blind** | `tone_metric/grpo_reward.py` |
| **SSIM** | voice-clone fidelity (ECAPA cosine) | `tone_metric/grpo_reward.py` |
| **len_ratio** | runaway / EOS-failure guard | duration model |

Wavs + a results JSON are uploaded to `s3://codec-audio-data/tts_data/yoruba/omnivoice_probe/`
for a **native ear** — which overrides the automatic gate.

### Decision rule (read with the verdict cell, §8)
- **GO** — `tone_i2` clears chance (~0.33) by ≥0.04 at coverage ≥0.70 **and** CER is sane (well
  below OmniVoice's reported Yoruba CER **21.37**): OmniVoice already carries real tone zero-shot →
  a finetune (`examples/run_finetune.sh`: Qwen3-0.6B, 8 codebooks, lr 1e-5, 5000 steps, 1×A100) is
  worth attempting.
- **NO-GO-FLAT** — `tone_i2` ≈ chance (monotone): same failure class as the **collapsed Qwen3-TTS
  base**. This is a **data-side** problem (clean tone-marked native audio), not a base-model problem;
  pivoting alone won't install tone.

### Caveats baked in
- OmniVoice's Yoruba pretraining is only **15.66 h** — its *weakest* of our 6 targets
  (Swahili 1.37 / Zulu 2.03 / Hausa 3.03 / Igbo 7.40 CER are strong).
- **Yoruba runtime language code is `yo`** (two-letter), **NOT `yor`** (`yor` is only the ISO-639-3
  column in OmniVoice's lang map).
- The minimal-pair set is **DRAFT** (`tone_metric/minimal_pairs_draft.json`, `verified:false`) — glosses
  from the literature; a native speaker must confirm before any benchmark is frozen.
- ⚠️ **Flywheel/license:** OmniVoice bundles the **Higgs/Boson** codec under a *non-commercial,
  output-restriction* license. If OmniVoice output is reused as synthetic training data (the project's
  flywheel), that output may be contaminated. Resolve before adopting OmniVoice as a data generator.
  (Model weights are Apache-2.0; the codec license governs generated audio.) See `README.md`.

**GPU: T4 (16 GB) or L4 (24 GB) — inference only.** Not the A100 (reserved for training).
sdpa/eager attention only; never install flash-attn.

## 1. Install + restart

Pin `numpy<2` (the gate modules want it), then install `omnivoice` (it pulls torch/torchaudio +
`transformers>=5.3`), then the scoring extras. The kernel **force-restarts** so the numpy pin takes
effect — after it restarts, run from §2 (the next code cell).

> **CONFIRM-AT-RUNTIME — transformers version.** OmniVoice declares `transformers>=5.3.0`; the
> `approachB` gate stack was built against `transformers>=4.46`. MMS-1b + AfriHuBERT should still load
> under 5.3, but if model loading breaks in §4, that is the first suspect. Fallback: generate the wavs
> here (cells through §6), then **score in a second session** with a transformers pin both stacks
> tolerate (the wavs are already on S3).

In [ ]:
%pip install --quiet "numpy<2"
# OmniVoice (real pip package; pulls torch/torchaudio>=2.4 and transformers>=5.3).
# If you hit an OmniVoice import/CUDA error, uncomment the pinned torch wheel and re-run:
# %pip install --quiet torch==2.8.0+cu128 torchaudio==2.8.0+cu128 --extra-index-url https://download.pytorch.org/whl/cu128
%pip install --quiet omnivoice
# Scoring stack for approachB modules. Do NOT pin transformers here (let omnivoice's pin stand).
%pip install --quiet boto3 soundfile soxr librosa speechbrain "swift-f0" pyworld accelerate safetensors "huggingface_hub>=0.24.0" tqdm
# Remove the Xet download backend: on a FRESH Colab runtime hf-xet stalls indefinitely at
# "Fetching N files: 0%" / "(incomplete total...) 0.00B" (documented huggingface_hub #3960/#4085 —
# this is the real cause of the §4 load hang, NOT a GPU/threading issue). Without the package,
# huggingface_hub falls back to the plain HTTP backend. Removing the package is the RELIABLE Xet
# kill: the HF_HUB_DISABLE_XET env var is read at import time and is unreliable once transformers/
# huggingface_hub get imported downstream (§2/§3), so we uninstall it outright here, pre-restart.
%pip uninstall -y hf-xet
import os
print("Installs done. RESTARTING kernel so numpy<2 takes effect — "
      "after it restarts, run from the NEXT cell (§2).", flush=True)
os._exit(0)   # force restart; if Colab does not auto-restart: Runtime > Restart session

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), \
    "numpy<2 pin did not take — re-run the install cell and let the kernel restart again"
print("numpy", np.__version__, "OK — continue.")

## 2. Clone OmniVoice (inference) + `mosesdaudu001/tone-on-a-budget` (approachB tone gate)

OmniVoice is cloned so we can (a) **introspect its real API at runtime** (the `generate()` signature
and the Yoruba code) and (b) fall back to the CLI if the class API drifts. The `approachB` tone modules
come from the private `mosesdaudu001/tone-on-a-budget` repo and go on `sys.path`.

> **Project rule.** The running notebook only sees `approachB` code already pushed to GitHub `main`.
> Edit → commit → push **before** re-running the clone cell, or Colab pulls the old version.

In [ ]:
import os, subprocess, sys, shutil

OV_DIR = "/content/OmniVoice" if os.path.isdir("/content") else "/tmp/OmniVoice"
if os.path.exists(OV_DIR): shutil.rmtree(OV_DIR)
subprocess.run(["git", "clone", "--depth", "1",
                "https://github.com/k2-fsa/OmniVoice.git", OV_DIR], check=True)

# --- Self-verify the API surface this notebook was written against (don't trust this file blindly) ---
ex = os.path.join(OV_DIR, "examples")
print("examples/ :", sorted(os.listdir(ex)) if os.path.isdir(ex) else "(missing)")
print("  expect: ['README.md','config','run_emilia.sh','run_eval.sh','run_finetune.sh']\n")

# Yoruba runtime code MUST be 'yo' (NOT 'yor'). Source of truth = docs/lang_id_name_map.tsv.
tsv = os.path.join(OV_DIR, "docs", "lang_id_name_map.tsv")
yo_ok = False
if os.path.exists(tsv):
    for ln in open(tsv):
        c = ln.rstrip("\n").split("\t")
        if (c and c[0] == "yo") or (len(c) > 1 and c[1].lower() == "yoruba"):
            print("Yoruba lang line:", ln.rstrip())   # expect: yo  Yoruba  yor  15.66
            yo_ok = c[0] == "yo"
else:
    print("TODO/CONFIRM: docs/lang_id_name_map.tsv not at expected path — find the lang map "
          "before trusting language='yo'.")
assert yo_ok, "Yoruba code is not 'yo' in the TSV — STOP and fix LANG_CODE in §3 before synthesizing."

# Show the real generate() signature so we use real arg names (text/language/ref_text/ref_audio/...).
sig = subprocess.run(["grep", "-n", "-A", "16", "def generate",
                      os.path.join(OV_DIR, "omnivoice", "models", "omnivoice.py")],
                     capture_output=True, text=True).stdout
print("\n=== generate() signature in source ===\n", sig[:1400] or "(grep found nothing — locate generate())")

In [ ]:
# tone-metric package (public) — replaces the old git clone + sys.path of tone_metric/
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
import os, subprocess, sys, shutil, importlib
importlib.invalidate_caches()
import tone_metric
from tone_metric import tone_probe as tp   # I1 (AfriHuBERT probe)
from tone_metric import tone_f0_abs as f0a   # I2 (F0-absolute tone meter)
from tone_metric.grpo_reward import RewardModels   # CER (MMS-1b-all yor) + SSIM (ECAPA)
CODE_DIR = os.path.dirname(tone_metric.__file__)   # minimal_pairs_draft.json ships as package data
print("tone_metric", tone_metric.__version__, "from", CODE_DIR)


## 3. Secrets + S3 + constants

`_secret()` resolves Colab `userdata` → `os.environ` → `getpass` (values never printed). Bucket
`codec-audio-data`, region `us-east-1`. All S3 keys below are taken verbatim from the project's
frozen artifacts (nb23/nb27/nb29).

In [ ]:
import os, getpass, boto3, random, torch, numpy as np

def _secret(k):
    try:
        from google.colab import userdata
        v = userdata.get(k)
        if v: return v
    except Exception:
        pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")

os.environ["AWS_ACCESS_KEY_ID"]     = _secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = _secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
HF_TOKEN = _secret("HF_TOKEN"); os.environ["HF_TOKEN"] = HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)

BUCKET = "codec-audio-data"
s3 = boto3.client("s3", region_name=os.environ["AWS_DEFAULT_REGION"])
s3.head_bucket(Bucket=BUCKET); print("S3 connected:", BUCKET)

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEED      = 4242                      # match nb26/nb27 seed family
SR        = 24000                     # OmniVoice output SR (prefer model.sampling_rate at runtime)
LANG_CODE = "yo"                      # VERIFIED: Yoruba runtime code is 'yo', NOT 'yor'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# S3 layout (verbatim from project artifacts)
S3_TONE_PREFIX = "tts_data/yoruba/tone_v2"
F0CAL_KEY      = "tts_data/yoruba/tone_v2/f0_abs_calibration.v1.json"   # nb27 calibration
HOLDOUTS_KEY   = "tts_data/yoruba_gold/holdouts.v1.json"               # 200 held-out eval texts
BIBLE_MANIFEST = "tts_data/yoruba_gold/s1_train.v2.jsonl"              # clean BibleTTS ref source
OUT_PREFIX     = "tts_data/yoruba/omnivoice_probe"                     # NEW: where this probe writes

WORK = "/content/ov_probe" if os.path.isdir("/content") else "/tmp/ov_probe"
os.makedirs(WORK, exist_ok=True)
print("DEVICE", DEVICE, "| SEED", SEED, "| OUT_PREFIX", OUT_PREFIX)

## 4. Load OmniVoice + the scoring stack

Load **both** up front so any `transformers`-version incompatibility surfaces now, before we spend
time synthesizing. Scoring stack = `RewardModels` (MMS-1b-all `yor` + ECAPA), AfriHuBERT encoder +
the tone probe `.pt` (latest from S3), and the frozen F0 calibration JSON.

In [ ]:
# === Robust OmniVoice loader — fixes the "Fetching 13 files: 0%" hang =============
# WHAT HANGS (verified against omnivoice/models/omnivoice.py @ k2-fsa/OmniVoice):
#   OmniVoice.from_pretrained(repo) -> _resolve_model_path(repo) ->
#       huggingface_hub.snapshot_download(repo)      # <- this is what prints "Fetching 13 files"
#   The hang is in the DOWNLOAD, before any weight reaches the GPU. So the Colab agent's
#   tqdm.thread_map / "moving weights to GPU" theory was WRONG: OmniVoice has zero thread_map and
#   does no manual GPU move, and patching thread_map left the hang at the exact same line.
# ROOT CAUSE (ranked): (1) hf-xet transport stalls on a fresh Colab runtime — the
#   "(incomplete total...) 0.00B / Fetching N files: 0%" signature is the documented Xet failure
#   (huggingface_hub #3960/#4085); (2) multithreaded snapshot_download contention; (3) no finite
#   download timeout; (4) stale .locks/*.incomplete from this session's interrupted retries.
#   "Worked first time" = a prior session whose 2.45 GB was already cached; this fresh runtime must
#   re-pull. (local_files_only=True did nothing: from_pretrained forwards **kwargs to
#   super().from_pretrained, never to snapshot_download, so the network fetch fires regardless.)
# FIX (defense-in-depth, all 4 causes): hf-xet removed in §1 (the key lever) + disable Xet/timeouts
#   here + wipe stale partial-download state + pre-download ONCE single-threaded with retries +
#   load from the LOCAL dir so from_pretrained skips snapshot_download entirely. The codec
#   (audio_tokenizer/) is bundled in the repo, so one snapshot covers everything (asserted below).
import os, shutil, glob, gc, torch

# Primary Xet kill is `pip uninstall -y hf-xet` in §1 (Xet is the real culprit on fresh Colab).
# These env vars are belt-and-suspenders (read at HF import time, which already happened upstream):
os.environ["HF_HUB_DISABLE_XET"]        = "1"      # route around the Xet transport stall
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"      # also disable the older hf_transfer accelerator
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]   = "60"     # default 10s; turns a dead socket into a retryable raise
os.environ["HF_HUB_ETAG_TIMEOUT"]       = "60"     # bound the metadata/HEAD revision check
try:                                               # best-effort in-process Xet off (if hf-xet lingers)
    import huggingface_hub.constants as _hc; _hc.HF_HUB_DISABLE_XET = True
except Exception:
    pass

REPO     = "k2-fsa/OmniVoice"
HUB      = os.path.join(os.path.expanduser(os.environ.get("HF_HOME", "~/.cache/huggingface")), "hub")
MODELDIR = os.path.join(HUB, "models--" + REPO.replace("/", "--"))

# 1) clear corrupt partial-download state so a resume cannot deadlock at 0%
locks = os.path.join(HUB, ".locks")
if os.path.isdir(locks):
    shutil.rmtree(locks, ignore_errors=True); print("cleared stale .locks")
inc = glob.glob(os.path.join(MODELDIR, "blobs", "*.incomplete"))
for f in inc:
    try: os.remove(f)
    except OSError: pass
if inc: print(f"removed {len(inc)} *.incomplete blob(s)")

# 2) drop any half-initialised model + free GPU left by earlier failed attempts
for _v in ("ov", "model"):
    globals().pop(_v, None)
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

# 3) pre-download single-threaded (max_workers=1 = no Colab thread deadlock) with retries;
#    the finite timeout above means a stall errors out and we retry, instead of hanging forever.
from huggingface_hub import snapshot_download
OV_LOCAL, _last = None, None
for _k in range(1, 4):
    try:
        OV_LOCAL = snapshot_download(REPO, max_workers=1, etag_timeout=60)   # resumable, single-threaded
        print(f"snapshot ready: {OV_LOCAL}"); break
    except Exception as e:
        _last = e; print(f"download attempt {_k}/3 failed: {type(e).__name__}: {e}")
if OV_LOCAL is None:
    raise RuntimeError(f"could not download {REPO} after 3 tries — just re-run this cell") from _last

# codec rides inside the snapshot -> from_pretrained finds it locally (no 2nd network fetch).
assert os.path.isdir(os.path.join(OV_LOCAL, "audio_tokenizer")), (
    "audio_tokenizer/ not in the snapshot — from_pretrained would now fetch "
    "eustlb/higgs-audio-v2-tokenizer over the network; pre-download it too with snapshot_download")

# 4) load from the LOCAL dir -> _resolve_model_path takes its os.path.isdir branch and never
#    calls snapshot_download again (the bundled audio_tokenizer/ codec also loads locally).
from omnivoice import OmniVoice
ov = OmniVoice.from_pretrained(
    OV_LOCAL,
    device_map=("cuda:0" if DEVICE == "cuda" else "cpu"),
    dtype=torch.float16,
)
OV_SR = int(getattr(ov, "sampling_rate", 0) or SR)   # cli/infer.py prefers model.sampling_rate
print("OmniVoice loaded; sampling_rate =", OV_SR)
assert OV_SR == 24000, f"CONFIRM: expected 24000 Hz, got {OV_SR} — adjust scoring resample if different"

In [ ]:
import json
from transformers import AutoModel, AutoFeatureExtractor

# (a) RewardModels: MMS-1b-all (yor adapter) + ECAPA. SSIM needs ECAPA.
rm = RewardModels(device=DEVICE)
assert rm.ecapa is not None, "ECAPA failed to load — SSIM needs it; check the speechbrain install"

# (b) AfriHuBERT encoder + feature extractor (I1 probe backbone)
enc = AutoModel.from_pretrained("ajesujoba/AfriHuBERT").to(DEVICE).eval()
fe  = AutoFeatureExtractor.from_pretrained("ajesujoba/AfriHuBERT")

# (c) Tone probe .pt — latest object under tts_data/yoruba/tone_v2/tone_probe_L*
objs = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{S3_TONE_PREFIX}/tone_probe_L").get("Contents")
assert objs, f"no tone_probe_L* under s3://{BUCKET}/{S3_TONE_PREFIX}/ (run nb22 first)"
probe_key = sorted(o["Key"] for o in objs)[-1]
s3.download_file(BUCKET, probe_key, f"{WORK}/tone_probe.pt")
PROBE, PMETA = tp.load_probe(f"{WORK}/tone_probe.pt", DEVICE)   # returns (probe, meta_dict)
LAYER = PMETA.get("layer", 9)
print("probe:", probe_key, "| layer:", LAYER)

# (d) F0-absolute calibration (theta_h/theta_l frozen by nb27)
s3.download_file(BUCKET, F0CAL_KEY, f"{WORK}/f0cal.json")
F0CAL = json.load(open(f"{WORK}/f0cal.json"))
I2_TH, I2_TL = F0CAL["theta_h"], F0CAL["theta_l"]
I2_MODE, I2_LATE = F0CAL.get("mode", "blind"), F0CAL.get("late_frac", 0.5)
print(f"F0 cal: theta_h={I2_TH} theta_l={I2_TL} mode={I2_MODE} late_frac={I2_LATE}")

## 5. Build the Yoruba probe set + fetch one clean reference

- **Minimal pairs:** `tone_metric/minimal_pairs_draft.json` realized in the first carrier
  `Mo rí ___ ní ọjà.` (same as nb24c) — these isolate **tone** (e.g. `ọkọ́` hoe / `ọkọ̀` vehicle /
  `ọkọ` husband).
- **Held-out lines:** a handful of SLR86 eval texts from `holdouts.v1.json["eval_texts"]` (disjoint
  from training) — these give a cleaner **CER**/naturalness read than the short carriers.
- **Reference (for cloning):** one clean single-speaker BibleTTS row from `s1_train.v2.jsonl`
  (`source=="bible"`): its wav (`audio_s3_key`) + transcript (`text`).

In [ ]:
import io, json, random, soundfile as sf

# --- (a) minimal pairs (authoritative draft set) ---
MP = json.load(open(os.path.join(CODE_DIR, "minimal_pairs_draft.json")))
CARRIER = MP["carriers"][0]["template"]                 # "Mo rí ___ ní ọjà."  (nb24c uses the first)
probe_lines = []                                        # dict(id, text, kind, base, word, tones)
for s in MP["sets"]:
    for j, it in enumerate(s["items"]):
        probe_lines.append(dict(
            id=f"mp_{s['base']}_{j}", kind="minpair", base=s["base"],
            word=it["text"], tones=it.get("tones"),
            text=CARRIER.replace("___", it["text"])))

# --- (b) a few held-out SLR86 eval lines ---
s3.download_file(BUCKET, HOLDOUTS_KEY, f"{WORK}/holdouts.json")
HOLD = json.load(open(f"{WORK}/holdouts.json"))
EVAL_ALL = list(HOLD["eval_texts"])                     # 200 rows: {base, text, zip_part}
rng = random.Random(SEED)
N_LONG = 8                                              # a handful; bump for broader CER coverage
for k, e in enumerate(rng.sample(EVAL_ALL, N_LONG)):
    probe_lines.append(dict(id=f"long_{k}_{e['base']}", kind="long", base=e["base"],
                            word=None, tones=None, text=e["text"]))
n_mp = sum(p["kind"] == "minpair" for p in probe_lines)
print(f"probe lines: {len(probe_lines)}  ({n_mp} minpair + {N_LONG} long)")

# --- (c) ONE clean BibleTTS reference: first source=='bible' row, 3-10 s (good clone-prompt length) ---
ref_row = None
body = s3.get_object(Bucket=BUCKET, Key=BIBLE_MANIFEST)["Body"]
for raw in io.TextIOWrapper(body, encoding="utf-8"):
    r = json.loads(raw)
    if r.get("source") == "bible" and 3.0 <= float(r.get("duration_sec", 0)) <= 10.0:
        ref_row = r; break
assert ref_row is not None, "no suitable bible ref row (3-10 s) in s1_train.v2.jsonl"
REF_WAV_KEY  = ref_row["audio_s3_key"]                  # tts_data/yoruba_gold/wav_biblefull/bible_NNNNN.wav
REF_TEXT     = ref_row["text"]                          # NFC, tone marks intact
REF_WAV_PATH = f"{WORK}/ref.wav"
s3.download_file(BUCKET, REF_WAV_KEY, REF_WAV_PATH)
ref_wav, ref_sr = sf.read(REF_WAV_PATH, dtype="float32")
if ref_wav.ndim > 1: ref_wav = ref_wav.mean(-1)
REF_DUR = len(ref_wav) / ref_sr
print(f"ref: {REF_WAV_KEY}  | {REF_DUR:.1f}s | {REF_TEXT[:64]}...")

## 6. Zero-shot synthesize with OmniVoice (Yoruba)

`language="yo"`, `ref_audio` = the clean bible wav (path), `ref_text` = its transcript. `generate()`
returns `list[np.ndarray]` at 24 kHz; take `audio[0]`. The first line is synthesized eagerly so any
API mismatch surfaces immediately; later failures are recorded and skipped.

In [ ]:
import soundfile as sf, numpy as np
from tqdm.auto import tqdm

syn, fails = [], []
for i, p in enumerate(tqdm(probe_lines, desc="OmniVoice synth")):
    try:
        audio = ov.generate(
            text=p["text"],
            language=LANG_CODE,          # "yo"  (verified; "yor" would fall back to None)
            ref_audio=REF_WAV_PATH,      # path str accepted; or (tensor, sr) tuple
            ref_text=REF_TEXT,
        )                                 # -> list[np.ndarray], shape (T,) @ OV_SR
        w = np.asarray(audio[0], dtype="float32")
        local = f"{WORK}/{p['id']}.wav"; sf.write(local, w, OV_SR)
        syn.append(dict(**p, wav=w, fs=OV_SR, local=local))
    except Exception as ex:
        fails.append((p["id"], f"{type(ex).__name__}: {ex}"))
        if i == 0:    # first line failing => API mismatch; surface it, don't silently skip everything
            raise
print(f"synthesized {len(syn)} clips; {len(fails)} failed")
for fid, msg in fails[:5]: print("  FAIL", fid, "->", msg)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def _up(rec):
    key = f"{OUT_PREFIX}/wav/{rec['id']}.wav"
    s3.upload_file(rec["local"], BUCKET, key); return key

with ThreadPoolExecutor(max_workers=16) as ex:
    up_keys = list(tqdm(ex.map(_up, syn), total=len(syn), desc="upload wavs"))
s3.upload_file(REF_WAV_PATH, BUCKET, f"{OUT_PREFIX}/ref/{os.path.basename(REF_WAV_KEY)}")
print(f"uploaded {len(up_keys)} wavs + ref -> s3://{BUCKET}/{OUT_PREFIX}/wav/")

## 7. Score each wav (CER / tone_i2 / tone_i2_cov / i1_acc / ssim / len_ratio)

Mirrors nb26's `dashboard_eval`: **one** shared MMS-1b forward (`rm.asr_logits`) feeds CER decode,
the I1 probe windows, and the I2 F0 windows.

- `cer`         — `RewardModels.cer(text, rm.decode_logits(logits))` (tone-blind)
- `tone_i2`     — balanced H/M/L accuracy from `f0_abs_score(...)` pred/target pairs
- `tone_i2_cov` — `f0_abs_score(...)["coverage"]` (per-TBU)
- `i1_acc`      — `probe_score(...)["accuracy"]` (quality/anti-collapse)
- `ssim`        — `rm.ssim(gen, fs, ref_wav, ref_sr)` (needs the clone ref)
- `len_ratio`   — `gen_dur / (0.157·len(text))` (the project duration model; runaway guard)

In [ ]:
import numpy as np

def _bal_tone_acc(pairs):
    """Mean over present classes of per-class recall (correct_c/covered_c). Chance ~1/3;
    an all-Mid prediction cannot game it. == nb26's _bal_tone_acc."""
    if not pairs: return float("nan")
    from collections import defaultdict
    tot, cor = defaultdict(int), defaultdict(int)
    for pp, tt in pairs:
        tot[tt] += 1; cor[tt] += int(pp == tt)
    recs = [cor[c] / tot[c] for c in tot if tot[c] > 0]
    return float(np.mean(recs)) if recs else float("nan")

def score_one(rec):
    wav, fs, text = rec["wav"], rec["fs"], rec["text"]
    logits, n16 = rm.asr_logits(wav, fs)                            # ONE shared MMS forward
    cer = RewardModels.cer(text, rm.decode_logits(logits))
    i1 = tp.probe_score(wav, fs, text, PROBE, enc, fe, asr=rm.asr, proc=rm.asr_proc,
                        device=DEVICE, emissions=logits, n16=n16, layer=LAYER)
    i2 = f0a.f0_abs_score(wav, fs, text, asr=rm.asr, proc=rm.asr_proc, device=DEVICE,
                          emissions=logits, n16=n16, theta_h=I2_TH, theta_l=I2_TL,
                          mode=I2_MODE, mid_ref=None, late_frac=I2_LATE)
    pairs = [(pp, tt) for pp, tt in zip(i2["pred"], i2["target"]) if pp is not None]
    return dict(
        id=rec["id"], kind=rec["kind"], word=rec.get("word"), tones=rec.get("tones"), text=text,
        cer=cer, i1_acc=i1["accuracy"], i1_cov=i1["coverage"],
        tone_i2=_bal_tone_acc(pairs), tone_i2_cov=i2["coverage"], tone_i2_bk=i2.get("backend"),
        ssim=rm.ssim(wav, fs, ref_wav, ref_sr),
        len_ratio=(len(wav) / fs) / max(0.157 * len(text), 1e-6),
        i2_pairs=pairs)

scored = [score_one(r) for r in tqdm(syn, desc="score")]
print("scored", len(scored), "clips")

In [ ]:
def _agg(rows):
    val = lambda k: [r[k] for r in rows if r[k] == r[k]]          # drop NaN
    allpairs = [pr for r in rows for pr in r["i2_pairs"]]
    return dict(
        n=len(rows),
        cer=float(np.mean(val("cer"))) if val("cer") else float("nan"),
        i1_acc=float(np.mean(val("i1_acc"))) if val("i1_acc") else float("nan"),
        i1_cov=float(np.mean([r["i1_cov"] for r in rows])) if rows else float("nan"),
        tone_i2=_bal_tone_acc(allpairs),
        tone_i2_cov=float(np.mean([r["tone_i2_cov"] for r in rows])) if rows else float("nan"),
        ssim=float(np.mean(val("ssim"))) if val("ssim") else float("nan"),
        ssim_min=float(np.min(val("ssim"))) if val("ssim") else float("nan"),
        len_ratio=float(np.mean([r["len_ratio"] for r in rows])) if rows else float("nan"))

agg_all  = _agg(scored)
agg_mp   = _agg([r for r in scored if r["kind"] == "minpair"])
agg_long = _agg([r for r in scored if r["kind"] == "long"])

hdr = f"{'subset':9} {'n':>3} {'cer':>6} {'toneI2':>7} {'I2cov':>6} {'I1acc':>6} {'ssim':>6} {'lenR':>6}"
print(hdr); print("-" * len(hdr))
for nm, a in [("ALL", agg_all), ("minpair", agg_mp), ("long", agg_long)]:
    print(f"{nm:9} {a['n']:>3} {a['cer']:>6.3f} {a['tone_i2']:>7.3f} {a['tone_i2_cov']:>6.3f} "
          f"{a['i1_acc']:>6.3f} {a['ssim']:>6.3f} {a['len_ratio']:>6.3f}")

print("\nAnchors (context, not live): OmniVoice reported Yoruba CER (paper/card) = 21.37 (>GT 17.97).")
print("Collapsed Qwen3-TTS base tone_i2 hovered ~0.398 (chance ~0.333). tone_i2 >> chance => real tone.")

## 8. Verdict — go / no-go

We apply an **absolute** version of nb26's `eligible()` (no live Qwen3-TTS `BASE_ROW` is loaded in
this cheap probe, so the `≥ BASE − δ` clauses become absolute anchors). If you *do* have the collapsed
base's row cached, swap the anchors for `ssim ≥ BASE['ssim']−0.05` and
`tone_i2 ≥ BASE['tone_i2']+0.04` to make it apples-to-apples.

In [ ]:
import json, datetime

TONE_CHANCE, TONE_MARGIN = 1/3, 0.04        # nb26's real-tone climb margin, over chance
SSIM_FLOOR = 0.40                           # absolute stand-in for BASE['ssim']-0.05
a = agg_all
checks = {
    "cer<=0.25":            a["cer"] <= 0.25,
    "i1_cov>=0.80":         a["i1_cov"] >= 0.80,
    "ssim>=0.40":           (a["ssim"] == a["ssim"]) and a["ssim"] >= SSIM_FLOOR,
    "len_ratio<=2.5":       a["len_ratio"] <= 2.5,
    "tone_i2_cov>=0.70":    a["tone_i2_cov"] >= 0.70,
    "tone_i2>=chance+0.04": (a["tone_i2"] == a["tone_i2"]) and a["tone_i2"] >= TONE_CHANCE + TONE_MARGIN,
}
PASS = all(checks.values())
flat = (a["tone_i2"] != a["tone_i2"]) or a["tone_i2"] < TONE_CHANCE + TONE_MARGIN

print("=== CHECKS ===")
for k, v in checks.items(): print(f"  [{'PASS' if v else 'FAIL'}] {k}")
print("\n=== VERDICT ===")
if PASS:
    verdict = "GO"
    print("GO: OmniVoice zero-shot Yoruba shows real tone + sane CER. A finetune "
          "(examples/run_finetune.sh: Qwen3-0.6B, 8 codebooks, lr 1e-5, 5000 steps, 1xA100) is worth it.\n"
          "NEXT: native ear on the uploaded wavs; THEN resolve the Boson-codec non-commercial license "
          "(flywheel contamination) before committing.")
elif flat:
    verdict = "NO-GO-FLAT"
    print("NO-GO-FLAT: tone_i2 ~ chance (monotone). This is a DATA-side problem, not a base-model "
          "problem — pivoting to OmniVoice will not by itself install tone (same failure class as the "
          "collapsed Qwen3-TTS base). Fix the data (clean tone-marked native audio) first.")
else:
    verdict = "MIXED"
    print("MIXED: some hard checks failed (see above). Inspect CER (intelligibility) and SSIM "
          "(cloning) before any finetune. The native ear is decisive.")

out = dict(
    date=datetime.date.today().isoformat(), seed=SEED, lang_code=LANG_CODE, ov_sampling_rate=OV_SR,
    ref_wav_key=REF_WAV_KEY, ref_text=REF_TEXT, probe_key=probe_key, probe_layer=LAYER,
    f0_cal=dict(theta_h=I2_TH, theta_l=I2_TL, mode=I2_MODE, late_frac=I2_LATE),
    agg_all=agg_all, agg_minpair=agg_mp, agg_long=agg_long, checks=checks, verdict=verdict,
    n_synth=len(scored), n_fail=len(fails),
    per_clip=[{k: v for k, v in r.items() if k != "i2_pairs"} for r in scored],
    wav_prefix=f"s3://{BUCKET}/{OUT_PREFIX}/wav/")
RESULTS_KEY = f"{OUT_PREFIX}/results.v1.json"
s3.put_object(Bucket=BUCKET, Key=RESULTS_KEY,
              Body=json.dumps(out, ensure_ascii=False, indent=2).encode("utf-8"))
print(f"\nresults -> s3://{BUCKET}/{RESULTS_KEY}")

## 9. (Optional) Strong-language sanity — fast flywheel preview

OmniVoice is strong on 5/6 project targets (Swahili CER 1.37, Zulu 2.03). Synthesizing 1–2 lines
confirms the install + cloning path and previews the contrast vs Yoruba's weak target. These are
**not** tone-gated (the gate is Yoruba-specific) — a smell test, and a "fast flywheel win" preview.

In [ ]:
# Confirm the codes against the cloned docs/lang_id_name_map.tsv (cell §2 printed Yoruba; grep others).
STRONG = [
    dict(code="sw", text="Habari ya asubuhi, karibu kwenye jaribio la sauti."),
    dict(code="zu", text="Sawubona, lokhu kuwukuhlolwa kwezwi."),
]
for s in STRONG:
    try:
        audio = ov.generate(text=s["text"], language=s["code"], ref_audio=REF_WAV_PATH, ref_text=REF_TEXT)
        w = np.asarray(audio[0], dtype="float32")
        loc = f"{WORK}/strong_{s['code']}.wav"; sf.write(loc, w, OV_SR)
        s3.upload_file(loc, BUCKET, f"{OUT_PREFIX}/strong/{s['code']}.wav")
        print(f"[{s['code']}] {len(w)/OV_SR:.1f}s -> s3://{BUCKET}/{OUT_PREFIX}/strong/{s['code']}.wav")
    except Exception as ex:
        print(f"[{s['code']}] FAILED ({type(ex).__name__}: {ex}) — confirm the code in the cloned TSV")
print("Strong-language smoke done. Listen on S3; expect cleaner output than Yoruba (far more train hours).")

### 10. Listen to generated Yoruba samples
Below are a few samples from the probe set (24kHz). These are the zero-shot outputs cloned from the BibleTTS reference.

In [ ]:
import IPython.display as ipd

# Pick a subset to display: 3 minimal pairs and 1 long sample
samples_to_show = []
for r in scored:
    if r['id'] in ['mp_oko_0', 'mp_oko_1', 'mp_oko_2', 'long_0_yom_02121_00124746688']:
        samples_to_show.append(r)

for s in samples_to_show:
    print(f"ID: {s['id']} | Kind: {s['kind']} | Text: {s['text']}")
    # Re-loading from the local file written in cell §6
    local_path = f"{WORK}/{s['id']}.wav"
    ipd.display(ipd.Audio(local_path))
    print("-" * 30)

## 11. (Follow-up) Instruction-channel probe — voice DESIGN, no reference clip

OmniVoice's `instruct=` arg does **voice design from a controlled set of attribute tags** (no
`ref_audio`) — **NOT free-form prose**. Valid English tags are a fixed vocabulary
(`omnivoice.utils.voice_design._INSTRUCT_VALID_EN`): gender (`male`/`female`), age (`child`,
`teenager`, `young adult`, `middle-aged`, `elderly`), pitch (`very low pitch` … `very high pitch`),
accents (`british accent`, `indian accent`, …), and `whisper`. Tags are comma-separated
(`"female, young adult, high pitch"`), one per mutually-exclusive group. This is voice *styling* —
NOT a PersonaPlex role/system prompt, and NOT a dialogue capability.

We deliberately use **gender + age + pitch only, no accent** — there is no African/Nigerian accent
in the vocab, and forcing a foreign accent (e.g. `british accent`) onto Yoruba would confound the
tone test.

This probe answers two things that matter for the **flywheel** (the TTS exists to manufacture training
data; for that we need *many distinct speakers*):
1. **Do the requested attributes come through for Yoruba?** (voice-design is English-trained, so unknown.)
2. **Does Yoruba tone survive WITHOUT a reference voice?** (cloning gave tone_i2 ≈ 0.60; does instruct?)

We synthesize a few attribute-tag voices × the minimal pairs + a couple long lines (no `ref_audio`),
score with the SAME tone gate (minus SSIM, meaningless with no clone target), and add a
**voice-diversity** check (are the tag-voices actually distinct?). The decisive read is still the **native ear**.

In [ ]:
# Voice DESIGN via instruct (no ref_audio). OmniVoice's instruct is NOT free-form prose — it is a
# comma-separated list of CONTROLLED attribute tags, validated against
# omnivoice.utils.voice_design._INSTRUCT_VALID_EN and split on re.split(r"\s*[,，]\s*", ...).
# We use gender+age+pitch only (NO accent: there is no African/Nigerian accent in the vocab, and
# forcing a foreign accent onto Yoruba would confound the tone test). Self-verify every tag first.
import re as _re
try:
    from omnivoice.utils.voice_design import _INSTRUCT_VALID_EN
    VALID = set(_INSTRUCT_VALID_EN)
except Exception:   # fallback to the vocab OmniVoice prints in its own validation error
    VALID = {"american accent","australian accent","british accent","canadian accent","child",
             "chinese accent","elderly","female","high pitch","indian accent","japanese accent",
             "korean accent","low pitch","male","middle-aged","moderate pitch","portuguese accent",
             "russian accent","teenager","very high pitch","very low pitch","whisper","young adult"}

INSTRUCTS = [                                   # 3 maximally-distinct, all-valid attribute voices
    "female, young adult, high pitch",
    "male, elderly, low pitch",
    "male, teenager, very high pitch",
]
for _d in INSTRUCTS:                            # fail loud here, not deep inside generate()
    _bad = [t for t in _re.split(r"\s*[,，]\s*", _d.lower()) if t and t not in VALID]
    assert not _bad, f"invalid instruct tag(s) {_bad} in '{_d}'. Valid English tags: {sorted(VALID)}"
print("instruct tags OK:", INSTRUCTS)

inst_lines = ([p for p in probe_lines if p["kind"] == "minpair"][:8]
              + [p for p in probe_lines if p["kind"] == "long"][:2])
syn_inst, inst_fail = [], []
for di, desc in enumerate(INSTRUCTS):
    for p in tqdm(inst_lines, desc=f"instruct[{di}]"):
        try:
            audio = ov.generate(text=p["text"], language=LANG_CODE, instruct=desc)   # NO ref_audio
            w = np.asarray(audio[0], dtype="float32")
            iid = f"inst{di}_{p['id']}"
            loc = f"{WORK}/{iid}.wav"; sf.write(loc, w, OV_SR)
            syn_inst.append(dict(id=iid, kind=p["kind"], text=p["text"], word=p.get("word"),
                                 tones=p.get("tones"), desc=desc, desc_i=di, wav=w, fs=OV_SR, local=loc))
        except Exception as ex:
            inst_fail.append((p["id"], di, f"{type(ex).__name__}: {ex}"))
            if di == 0 and p is inst_lines[0]:
                raise   # first call failing => instruct API/tag issue; surface it instead of skipping all
print(f"instruct clips: {len(syn_inst)}; failed: {len(inst_fail)}")
for fid, di, msg in inst_fail[:5]:
    print("  FAIL", fid, "desc", di, "->", msg)

In [ ]:
# Score: tone survival (tone_i2) + intelligibility (cer) + length sanity. SSIM-to-bible is meaningless
# here (we are NOT cloning that voice) -> instead measure VOICE DIVERSITY across the three descriptions.
def score_inst(rec):
    wav, fs, text = rec["wav"], rec["fs"], rec["text"]
    logits, n16 = rm.asr_logits(wav, fs)
    cer = RewardModels.cer(text, rm.decode_logits(logits))
    i2 = f0a.f0_abs_score(wav, fs, text, asr=rm.asr, proc=rm.asr_proc, device=DEVICE,
                          emissions=logits, n16=n16, theta_h=I2_TH, theta_l=I2_TL,
                          mode=I2_MODE, mid_ref=None, late_frac=I2_LATE)
    pairs = [(pp, tt) for pp, tt in zip(i2["pred"], i2["target"]) if pp is not None]
    return dict(id=rec["id"], desc_i=rec["desc_i"], kind=rec["kind"], cer=cer,
                tone_i2=_bal_tone_acc(pairs), tone_i2_cov=i2["coverage"],
                len_ratio=(len(wav) / fs) / max(0.157 * len(text), 1e-6), i2_pairs=pairs)

scored_inst = [score_inst(r) for r in tqdm(syn_inst, desc="score instruct")]

from concurrent.futures import ThreadPoolExecutor
def _upi(rec):
    k = f"{OUT_PREFIX}/instruct/{rec['id']}.wav"; s3.upload_file(rec["local"], BUCKET, k); return k
with ThreadPoolExecutor(max_workers=16) as ex:
    list(ex.map(_upi, syn_inst))

def _ai(rows):
    pr = [p for r in rows for p in r["i2_pairs"]]
    v = lambda k: [r[k] for r in rows if r[k] == r[k]]
    return dict(n=len(rows), cer=float(np.mean(v("cer"))) if v("cer") else float("nan"),
                tone_i2=_bal_tone_acc(pr),
                tone_i2_cov=float(np.mean([r["tone_i2_cov"] for r in rows])) if rows else float("nan"),
                len_ratio=float(np.mean([r["len_ratio"] for r in rows])) if rows else float("nan"))

print(f"{'mode/desc':26} {'n':>3} {'cer':>6} {'toneI2':>7} {'I2cov':>6} {'lenR':>6}")
print("-" * 60)
for di, desc in enumerate(INSTRUCTS):
    a = _ai([r for r in scored_inst if r["desc_i"] == di])
    print(f"{('instruct['+str(di)+']'):26} {a['n']:>3} {a['cer']:>6.3f} {a['tone_i2']:>7.3f} "
          f"{a['tone_i2_cov']:>6.3f} {a['len_ratio']:>6.3f}")
ai_all = _ai(scored_inst)
print(f"{'INSTRUCT all':26} {ai_all['n']:>3} {ai_all['cer']:>6.3f} {ai_all['tone_i2']:>7.3f} "
      f"{ai_all['tone_i2_cov']:>6.3f} {ai_all['len_ratio']:>6.3f}")
print(f"{'CLONE all (ref, from §7)':26} {agg_all['n']:>3} {agg_all['cer']:>6.3f} {agg_all['tone_i2']:>7.3f} "
      f"{agg_all['tone_i2_cov']:>6.3f} {agg_all['len_ratio']:>6.3f}")

# Voice diversity: for each line synthesized under >=2 descriptions, pairwise ECAPA SSIM across them.
# LOW mean = instruct yields DISTINCT voices (good); ~0.8+ = instruct is NOT changing the voice.
from itertools import combinations
by_line = {}
for r in syn_inst:
    by_line.setdefault(r["id"].split("_", 1)[1], {})[r["desc_i"]] = r["wav"]
divs = []
for d in by_line.values():
    for i, j in combinations(sorted(d), 2):
        s = rm.ssim(d[i], OV_SR, d[j], OV_SR)
        if s is not None:
            divs.append(s)
if divs:
    print(f"\nvoice diversity: mean pairwise SSIM across descriptions = {float(np.mean(divs)):.3f} "
          f"(LOWER = more distinct voices)")

### Read-out

Decisive check = the **native ear** on `s3://codec-audio-data/tts_data/yoruba/omnivoice_probe/instruct/`:
do the three tag-voices sound **distinct** and **match their requested attributes** (gender / age /
pitch), and is the Yoruba still **correct** (words + tone)?

- **INSTRUCT tone_i2 ≈ CLONE tone_i2** *and* low voice-diversity SSIM → instruct preserves Yoruba tone
  AND gives distinct voices → a **cheap speaker-diversity lever for the flywheel** (no reference clips
  needed per speaker).
- **INSTRUCT tone_i2 craters** vs CLONE, or voices are English-accented / identical → instruct does not
  carry Yoruba tone → use **reference-clip cloning** to mint flywheel speakers instead.

Caveat: voice-design is English-trained; whether it honors the **attribute tags** *and* keeps Yoruba
tone is exactly the open question this cell tests. (Flywheel use is still gated on the Boson-codec
license — see `README.md` / `BOSON_LICENSE_WAIVER_EMAIL.md`.)